# Fine-Tune Open Weight Language Models for Function Calling in Strands Agents

[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)

## Introduction

This notebook demonstrates how to fine-tune small language models (1-3B parameters) for reliable function calling in edge environments. We leverage optimized training pipelines to achieve 2-5x faster training with 50% less memory usage compared to standard implementations.

Function calling, also known as tool use, enables language models to interact with external systems through structured API calls. This is critical for edge deployment where models must control physical devices, query databases, or invoke services with high reliability.

### The Problem We're Solving

Edge devices in vehicles and industrial settings need AI assistants that can:
- **Understand natural language**: "It's too hot" → climate control
- **Execute actions locally**: No cloud dependency for critical controls
- **Work with limited resources**: 2-4GB RAM, no GPU required
- **Maintain high accuracy**: Safety-critical operations demand reliability

### Our Solution: Fine-Tuned Tool Calling

Instead of using a general-purpose model, we specialize Qwen3-1.7B for specific tools:

```mermaid
graph LR
    A[User Input:<br/>'Set the temperature to 72 degrees'] --> B[Model Recognition:<br/>Intent = climate_control<br/>Parameter = 72]
    B --> C[Strands Format:<br/>toolUse.name: climate_control<br/>toolUse.input.command: 'set to 72']
    C --> D[Agent Execution:<br/>Virtual ECU updates<br/>climate state]
    D --> E[User Feedback:<br/>'Temperature set to 72°F']
    
    style A fill:#e1f5fe
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#fce4ec
```

### What You'll Learn

- Generate high-quality synthetic training data using teacher-student approaches
- Fine-tune models efficiently using LoRA (Low-Rank Adaptation)
- Quantize models for edge deployment while preserving accuracy
- Benchmark performance improvements systematically
- Deploy models using llama.cpp for production inference

### Understanding Model Quantization

Quantization is the process of reducing numerical precision to compress models while preserving performance. In neural networks, weights and activations typically use 32-bit (FP32) or 16-bit (FP16/BF16) floating-point numbers. Quantization reduces these to 8-bit integers or even 4-bit representations, achieving:

- **Size Reduction**: 4-bit quantization reduces model size by ~75% compared to FP16
- **Speed Improvement**: Integer operations are faster than floating-point on most hardware
- **Memory Efficiency**: Enables deployment on resource-constrained edge devices
- **Minimal Accuracy Loss**: Modern quantization methods preserve >99% of model quality

#### Quantization Methods

**K-means Quantization (K-quants)**: Groups similar weights into clusters, storing cluster indices instead of full values. The 'K' variants (Q4_K_M, Q5_K_M) use different bit allocations for different tensor components.

**Dynamic vs Static**: Dynamic quantization determines scale factors at runtime, while static uses pre-computed values. We use static for predictable edge performance.

**Mixed Precision**: Critical layers (like embeddings) maintain higher precision while less sensitive layers use aggressive quantization.

## Install Dependencies & Build Tools

This section installs all required dependencies and builds llama.cpp for GGUF operations.
Run these cells in order - the complete process may take 5-10 minutes.

### What this installs:
- **Python packages**: PyTorch, transformers, PEFT, GGUF, etc.
- **llama.cpp**: Cloned and compiled for GGUF conversion and inference
- **System tools**: Build essentials, CMake, etc.

In [ ]:
# Update package lists and install all system dependencies
!sudo apt-get update -qq
!sudo apt-get install -y build-essential cmake git wget curl pkg-config libssl-dev libcurl4-openssl-dev

### Install Python Dependencies

Install all required Python packages using uv package manager.

In [ ]:
# Install Python dependencies
!cd ../../ && pip install -e .[dev,ml,function-calling]
!pip install ipywidgets trl  # trl for SFTTrainer

### Clone llama.cpp Repository

Clone the official llama.cpp repository for GGUF conversion and inference tools.

In [ ]:
# Clone llama.cpp if not already present
!if [ ! -d "./llama.cpp" ]; then \
    git clone https://github.com/ggerganov/llama.cpp.git; \
else \
    cd llama.cpp && git pull; \
fi

### Build llama.cpp with CUDA Support

Check for CUDA availability and compile llama.cpp with appropriate optimizations.

In [ ]:
# Build llama.cpp with CUDA support if available
!cd llama.cpp && rm -rf build && \
if command -v nvcc &> /dev/null; then \
    cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF; \
    cmake --build build --config Release -j $(nproc); \
else \
    cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF; \
    cmake --build build --config Release -j $(nproc); \
fi

### Verify Installation

Check that all components are properly installed and working.

In [ ]:
# Verify Python packages and llama.cpp binaries
from pathlib import Path

# Test key Python imports
try:
    import torch
    import transformers
    import peft
    import gguf
    print("Python packages: All required packages available")
except ImportError as e:
    print(f"Missing package: {e}")

# Check for critical llama.cpp files
llamacpp_dir = Path("./llama.cpp")
build_dir = llamacpp_dir / "build" / "bin"
key_files = [
    build_dir / "llama-server",
    build_dir / "llama-quantize", 
    llamacpp_dir / "convert_hf_to_gguf.py"
]

missing_files = [f for f in key_files if not f.exists()]
if missing_files:
    print(f"Missing files: {[str(f) for f in missing_files]}")
else:
    print("llama.cpp: All required binaries and scripts available")

## Environment Setup

Configure the training environment and verify GPU availability for optimal performance.

### Hardware Requirements
- **GPU Memory**: 8GB+ VRAM recommended for training
- **System RAM**: 16GB+ for data processing
- **Storage**: 10GB+ free space for models and intermediate files

The training uses mixed precision (FP16) and gradient checkpointing to optimize memory usage.

In [ ]:
# Imports
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Data Preparation

This section covers the preparation of high-quality training data for function calling. Our approach uses synthetic data generation to create diverse, realistic examples that cover all vehicle control scenarios.

### Training Data Characteristics

Our dataset includes:
- **1000 training examples** covering all 5 vehicle control domains
- **200 test examples** for validation and evaluation
- **Bilingual support**: English and Japanese language patterns
- **Realistic scenarios**: Natural language requests users would actually make

### Data Quality Features

1. **Diverse Phrasing**: Multiple ways to express the same intent
2. **Parameter Variation**: Different values and combinations for each tool
3. **Context Awareness**: Requests that consider vehicle state and user preferences
4. **Error Handling**: Examples of ambiguous requests and clarifications

### Synthetic Data Generation Process

The training data was generated using a teacher-student approach:
1. **Teacher Model**: Claude 3.7 Sonnet generates high-quality examples
2. **Tool Specifications**: Real vehicle control APIs define the output format
3. **Quality Control**: Automated validation ensures correct JSON structure
4. **Diversity Injection**: Systematic variation of parameters and phrasing

### Optional: Generate Custom Training Data

This notebook includes pre-generated training data (1000 examples) and test data (200 examples) that follows the correct format for function calling. However, if you want to generate additional training data or customize the examples, you can use the built-in data generator.

**Requirements for data generation:**
- AWS Bedrock access with Claude 3.7 Sonnet (for high-quality synthetic data)
- Proper AWS credentials configured
- Bearer token (if required by your setup)

**Skip this cell if you want to use the existing training data.**

### Data Generation Strategy

If you choose to generate custom data, the process follows these steps:
1. **Tool Analysis**: Extract function signatures and parameter types
2. **Scenario Generation**: Create realistic user scenarios for each tool
3. **Language Variation**: Generate multiple phrasings for each scenario
4. **Parameter Sampling**: Systematically vary parameter values
5. **Quality Validation**: Ensure all generated examples are valid JSON

The generator can create thousands of examples in minutes, but the pre-generated dataset is sufficient for most use cases.

In [ ]:
# # OPTIONAL: Generate additional training data using AWS Bedrock
# # Uncomment the lines below if you want to generate custom training data

# import os

# # Set your AWS Bearer Token
# os.environ['AWS_BEARER_TOKEN_BEDROCK'] = 'your_token_here'
# os.environ['AWS_REGION'] = 'us-east-1'

# # Generate training data
# from utils.data_generator import DataGenerator, ToolRegistry

# generator = DataGenerator()
# generator.generate_dataset(
#     num_examples=1000,
#     output_path="data/train.jsonl", 
#     output_format="conversations"
# )

# # Generate test data  
# generator.generate_dataset(
#     num_examples=200,
#     output_path="data/test.jsonl",
#     output_format="conversations"
# )

### Load Pre-Generated Training Data

Load the curated training and test datasets that have been optimized for function calling performance.

In [ ]:
# Load training data in SFTTrainer format
train_path = Path('data/train.jsonl')
test_path = Path('data/test.jsonl')

train_data = []
with open(train_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        train_data.append(data)

test_data = []
with open(test_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        test_data.append(data)

print(f"Train: {len(train_data)}, Test: {len(test_data)}")
print("Using SFT format data (messages + tools)")

### Exploratory Data Analysis

Analyze the characteristics of our training data to understand tool distribution, conversation structure, and data quality. The analysis adapts to both legacy text format and modern SFT format automatically.

#### SFT Format Analysis Features:
- **Tool Call Distribution**: Analyze which tools are used most frequently
- **Conversation Structure**: Number of messages per conversation 
- **Multi-Tool Usage**: Conversations that use multiple tools in sequence
- **Message Role Analysis**: Distribution of system/user/assistant/tool messages
- **Tool Call Accuracy**: Proper formatting of tool_calls arrays

#### Legacy Format Analysis Features:
- **Text Length Distribution**: Character count analysis of formatted text
- **Tool Usage Patterns**: Extracted from XML-style tool calls
- **Template Structure**: Consistency of chat template formatting

The analysis automatically detects the data format and provides relevant insights for training optimization.

In [ ]:
# Exploratory Data Analysis of SFT Format Training Data
import matplotlib.pyplot as plt
import json
from collections import Counter

def analyze_sft_dataset(data_list, dataset_name):
    """Analyze SFT format dataset (messages + tools)."""
    print(f"Analyzing {dataset_name} dataset with {len(data_list)} examples...")
    
    tool_counts = Counter()
    conversation_lengths = []
    message_counts = []
    tool_call_counts = []
    
    for item in data_list:
        messages = item.get('messages', [])
        tools = item.get('tools', [])
        
        message_counts.append(len(messages))
        
        # Count tool calls in assistant messages
        tool_calls_in_conversation = 0
        conversation_length = 0
        
        for msg in messages:
            # Calculate conversation length
            content = msg.get('content', '')
            if content:
                conversation_length += len(str(content))
            
            # Look for tool calls in assistant messages
            if msg.get('role') == 'assistant' and 'tool_calls' in msg:
                tool_calls = msg['tool_calls']
                if tool_calls:
                    tool_calls_in_conversation += len(tool_calls)
                    
                    # Count specific tools used
                    for tool_call in tool_calls:
                        if isinstance(tool_call, dict):
                            function = tool_call.get('function', {})
                            tool_name = function.get('name', 'unknown')
                            if tool_name != 'unknown':
                                tool_counts[tool_name] += 1
        
        tool_call_counts.append(tool_calls_in_conversation)
        conversation_lengths.append(conversation_length)
    
    return tool_counts, conversation_lengths, message_counts, tool_call_counts

# Analyze both datasets
print("Analyzing SFT format data...")
train_tools, train_lengths, train_msg_counts, train_tool_calls = analyze_sft_dataset(train_data, "Training")
test_tools, test_lengths, test_msg_counts, test_tool_calls = analyze_sft_dataset(test_data, "Test")

# Create visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Chart 1: Tool Distribution Comparison
tools = list(set(train_tools.keys()) | set(test_tools.keys()))
train_counts = [train_tools.get(tool, 0) for tool in tools]
test_counts = [test_tools.get(tool, 0) for tool in tools]

x = range(len(tools))
width = 0.35
ax1.bar([i - width/2 for i in x], train_counts, width, label='Training', alpha=0.8)
ax1.bar([i + width/2 for i in x], test_counts, width, label='Test', alpha=0.8)
ax1.set_xlabel('Tool Names')
ax1.set_ylabel('Count')
ax1.set_title('Tool Usage Distribution')
ax1.set_xticks(x)
ax1.set_xticklabels([tool.replace('_', '\n') for tool in tools], rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Chart 2: Conversation Length Distribution
ax2.hist(train_lengths, bins=30, alpha=0.7, label=f'Training (n={len(train_data)})', density=True)
ax2.hist(test_lengths, bins=30, alpha=0.7, label=f'Test (n={len(test_data)})', density=True)
ax2.set_xlabel('Conversation Length (chars)')
ax2.set_ylabel('Density')
ax2.set_title('Conversation Length Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Chart 3: Dataset Size Comparison
datasets = ['Training', 'Test']
sizes = [len(train_data), len(test_data)]
colors = ['#1f77b4', '#ff7f0e']
bars = ax3.bar(datasets, sizes, color=colors, alpha=0.8)
ax3.set_ylabel('Number of Examples')
ax3.set_title('Dataset Sizes')
ax3.grid(True, alpha=0.3)

# Add value labels on bars
for bar, size in zip(bars, sizes):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + max(sizes)*0.01,
             f'{size:,}', ha='center', va='bottom', fontweight='bold')

# Chart 4: Messages per Conversation
ax4.hist(train_msg_counts, bins=20, alpha=0.7, label='Training', density=True)
ax4.hist(test_msg_counts, bins=20, alpha=0.7, label='Test', density=True)
ax4.set_xlabel('Messages per Conversation')
ax4.set_ylabel('Density')
ax4.set_title('Conversation Structure Analysis')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed statistics
print(f"\n=== SFT Dataset Analysis Results ===")
print(f"Training: {len(train_data):,} examples")
print(f"Test: {len(test_data):,} examples")

# Calculate metrics
avg_msgs_train = sum(train_msg_counts) / len(train_msg_counts)
avg_msgs_test = sum(test_msg_counts) / len(test_msg_counts)
avg_tools_train = sum(train_tool_calls) / len(train_tool_calls)
avg_tools_test = sum(test_tool_calls) / len(test_tool_calls)

print(f"Average messages per conversation - Training: {avg_msgs_train:.1f}, Test: {avg_msgs_test:.1f}")
print(f"Average tool calls per conversation - Training: {avg_tools_train:.1f}, Test: {avg_tools_test:.1f}")
print(f"Conversation length - Training: {sum(train_lengths)/len(train_lengths):.0f} chars, Test: {sum(test_lengths)/len(test_lengths):.0f} chars")

# Multi-tool conversation analysis
multi_tool_train = sum(1 for count in train_tool_calls if count > 1)
multi_tool_test = sum(1 for count in test_tool_calls if count > 1)
print(f"Multi-tool conversations - Training: {multi_tool_train} ({multi_tool_train/len(train_data)*100:.1f}%), Test: {multi_tool_test} ({multi_tool_test/len(test_data)*100:.1f}%)")

# Tool distribution analysis
print(f"\nTool Usage Distribution:")
for tool in sorted(tools):
    train_count = train_tools.get(tool, 0)
    test_count = test_tools.get(tool, 0)
    total = train_count + test_count
    print(f"  {tool}: {total} total ({train_count} train, {test_count} test)")

## Model Setup

Configure the Qwen3-1.7B model for function calling fine-tuning. This section covers model loading, tokenizer setup, and LoRA configuration.

### Model Configuration Strategy

Our setup optimizes for:
1. **Memory Efficiency**: FP16 precision reduces memory usage by 50%
2. **Training Speed**: Gradient checkpointing enables larger batch sizes
3. **Stability**: Proper tokenizer configuration prevents training issues
4. **Compatibility**: Device mapping ensures GPU utilization when available

In [ ]:
# Load base model
model_name = "Qwen/Qwen3-1.7B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")

### Configure LoRA for Parameter-Efficient Fine-tuning

Apply LoRA (Low-Rank Adaptation) to enable efficient fine-tuning with minimal computational overhead.

#### LoRA Configuration Explained:
- **Rank (r=8)**: Controls adaptation capacity vs efficiency trade-off
- **Alpha (α=32)**: Scaling factor that amplifies LoRA contributions
- **Dropout (0.1)**: Regularization to prevent overfitting
- **Target Modules**: All attention and MLP layers for comprehensive adaptation

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.enable_input_require_grads()

## Training Configuration & Execution

Configure and execute the fine-tuning process using SFTTrainer with optimized hyperparameters for function calling performance.

### SFTTrainer vs Base Trainer

We now use **SFTTrainer** from the `trl` library instead of the base Transformer's `Trainer`. Key advantages:

#### **Chat Template Integration**
- SFTTrainer automatically applies the model's chat template during training
- No need to pre-format conversations into text strings
- Ensures training format matches inference format exactly

#### **Tool Calling Support**
- Native support for `messages` format with `tool_calls` arrays
- Processes tool responses with `role: "tool"` correctly
- Handles the `tools` schema definitions automatically

#### **Better Training Dynamics**
- Optimized loss calculation for conversational data
- Better handling of multi-turn conversations
- Improved convergence for instruction-following tasks

#### **Memory Efficiency**
- More efficient tokenization of conversation data
- Better gradient computation for chat formats
- Reduced memory overhead compared to text-based approaches

### Data Format Support

The updated pipeline supports both formats automatically:
- **SFT Format**: `{"messages": [...], "tools": [...]}` (recommended)  
- **Legacy Format**: `{"text": "..."}` (fallback)

SFTTrainer will automatically detect and handle the appropriate format.

In [ ]:
# Prepare datasets for SFTTrainer
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Test dataset: {len(test_dataset)} examples")

# Show example of SFT data format
print(f"\nExample training sample:")
example = train_dataset[0]
print(f"Keys: {list(example.keys())}")
print(f"Messages: {len(example['messages'])}")
print(f"Tools: {len(example['tools'])}")
print(f"Sample user message: {example['messages'][1]['content'][:100]}...")

# Check for tool calls
has_tool_calls = any('tool_calls' in msg for msg in example['messages'])
print(f"Contains tool calls: {has_tool_calls}")

### Configure Training Arguments

Set up optimized training parameters for function calling fine-tuning.

In [ ]:
# Training configuration for SFTTrainer

training_args = SFTConfig(
    output_dir="./qwen3-function-calling",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    max_length=1024,  # Maximum sequence length
    packing=False  # Don't pack sequences for tool calling
)

# Configure SFTTrainer for tool calling
print("Configuring SFTTrainer for tool calling...")
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
    # SFTTrainer automatically handles messages and tools format
)

print("SFTTrainer configured successfully")

### Execute Training Process

Start the fine-tuning process using SFTTrainer. This will take approximately 15-30 minutes depending on your GPU.

#### SFTTrainer Advantages:
1. **Chat Template Integration**: Automatically applies the model's chat template during training
2. **Tool Calling Support**: Native support for messages with tool_calls format
3. **Better Convergence**: Optimized for instruction/chat fine-tuning patterns
4. **Memory Efficiency**: Improved memory usage compared to base Trainer

#### What Happens During Training:
1. **Template Application**: SFTTrainer applies the chat template to format training examples
2. **Tool Call Processing**: Processes tool_calls in messages and converts to model tokens
3. **Loss Calculation**: Computes loss on assistant responses and tool calls only
4. **Parameter Update**: Updates LoRA parameters to improve tool calling accuracy
5. **Evaluation**: Validates on test set using the same chat template formatting

#### Training Progress Indicators:
- **Decreasing Loss**: Model is learning the function calling patterns
- **Stable Training**: SFTTrainer provides more stable training for chat formats
- **Tool Accuracy**: Better handling of structured tool calls vs. free-form text

The SFTTrainer will automatically handle the complexity of chat template formatting and tool calling structure, leading to better fine-tuning results for function calling tasks.

In [ ]:
# Train the model
trainer.train()
trainer.save_model("./qwen3-function-calling-final")
tokenizer.save_pretrained("./qwen3-function-calling-final")

## Training Results & Analysis

Analyze the training progress and visualize performance metrics to understand how well the model learned function calling patterns. The training process tracks several important metrics:

#### Loss Metrics:
- **Training Loss**: Measures how well the model fits the training data
- **Validation Loss**: Indicates generalization to unseen examples
- **Loss Convergence**: Steady decrease shows effective learning

#### Performance Indicators:
- **Gradient Norms**: Stable gradients indicate healthy training dynamics
- **Learning Rate Schedule**: Warmup and decay optimize convergence
- **Memory Usage**: Efficient utilization of available GPU memory

#### Expected Training Outcomes

For successful function calling fine-tuning:
- **Training Loss**: Should decrease from ~2.5 to ~0.5 over 3 epochs
- **Validation Loss**: Should follow training loss without large gaps
- **Convergence**: Loss should stabilize in the final epoch
- **No Overfitting**: Validation loss shouldn't increase while training loss decreases

In [ ]:
# Display training metrics
import matplotlib.pyplot as plt

# Extract loss from training history
train_loss = [log['loss'] for log in trainer.state.log_history if 'loss' in log]
eval_loss = [log['eval_loss'] for log in trainer.state.log_history if 'eval_loss' in log]

# Create loss plot
if train_loss or eval_loss:
    plt.figure(figsize=(10, 5))
    
    if train_loss:
        plt.subplot(1, 2, 1)
        plt.plot(train_loss)
        plt.title('Training Loss')
        plt.xlabel('Steps')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
    
    if eval_loss:
        plt.subplot(1, 2, 2)
        plt.plot(eval_loss, 'orange')
        plt.title('Evaluation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Model Export & Preparation

Export the fine-tuned model by merging LoRA weights with the base model for deployment. The fine-tuning process creates separate LoRA adapter weights that need to be merged with the base model:

#### Why Merge LoRA Weights?
1. **Deployment Simplicity**: Single model file instead of base + adapter
2. **Inference Speed**: No additional computation overhead during inference
3. **Compatibility**: Works with standard inference engines like llama.cpp
4. **Storage Efficiency**: Eliminates need to store both base model and adapters

#### Merge Process:
1. **Load LoRA Adapters**: Retrieve trained low-rank matrices
2. **Compute Full Weights**: Multiply and add to base model weights
3. **Create Merged Model**: New model with integrated adaptations
4. **Preserve Tokenizer**: Ensure consistent text processing

The merged model maintains all the improvements from fine-tuning while being ready for deployment.

In [ ]:
# Merge LoRA weights and save
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./qwen3-function-calling-merged")
tokenizer.save_pretrained("./qwen3-function-calling-merged")

## Chat Template Configuration

Configure the correct chat template for tool calling compatibility before GGUF conversion. The chat template defines how conversations are formatted for the model:

```mermaid
flowchart LR
    A[Fine-tuned<br/>Qwen3 Model] --> B[Template<br/>Injection]
    B --> C[GGUF<br/>Conversion]
    C --> D[Quantization<br/>Q4_K_M]
    D --> E[llama.cpp<br/>Server]
    E --> F[Strands<br/>SDK]
    
    style A fill:#f9f9f9,stroke:#333,stroke-width:2px
    style B fill:#e8f4f8,stroke:#333,stroke-width:2px
    style C fill:#f0f8e8,stroke:#333,stroke-width:2px
    style D fill:#e8e8f8,stroke:#333,stroke-width:2px
    style E fill:#f8f0e8,stroke:#333,stroke-width:2px
    style F fill:#f8e8f0,stroke:#333,stroke-width:2px
```

#### Stage 1: Fine-tuned Model
The LoRA-adapted Qwen3 model after training on function calling examples. At this stage, the model has learned the patterns but lacks the Jinja2 chat template required for proper message formatting.

#### Stage 2: Template Injection
The original Qwen3 chat template is extracted from the base model and injected into the fine-tuned model's tokenizer configuration. This template defines the conversation structure including system prompts, user messages, assistant responses, and tool definitions.

#### Stage 3: GGUF Conversion
The model is converted to GGUF format with the chat template embedded as metadata in the file header. This creates a self-contained binary containing both model weights and formatting instructions. The conversion preserves the template alongside the FP16 model weights.

#### Stage 4: Quantization
Model weights are compressed from FP16 to Q4_K_M format, reducing size by approximately 75% (3.4GB to 1.1GB). The quantization process only affects tensor weights while preserving all metadata including the chat template, special tokens, and model architecture information.

#### Stage 5: Server Deployment
The llama.cpp server loads the GGUF file and extracts the embedded chat template. It initializes a Jinja2 engine to process incoming OpenAI-compatible API requests, applying the template to format messages and tool definitions correctly for model inference.

### Template Injection Process

We extract the original Qwen3 chat template and apply it to our fine-tuned model to ensure compatibility with Strands agents and llama.cpp inference.

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

# The original Qwen3 model has a proper chat template that supports tool calling
# We need to get this template and apply it to our merged model

# Load the original Qwen3 tokenizer to get the correct chat template
original_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B', trust_remote_code=True)

# Get the correct chat template
correct_chat_template = original_tokenizer.chat_template

# Verify it supports function calling
try:
    test_messages = [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'Hello'}
    ]
    test_tools = [{
        'type': 'function',
        'function': {
            'name': 'test_tool',
            'description': 'A test tool',
            'parameters': {'type': 'object', 'properties': {}}
        }
    }]
    
    # Test with tools
    formatted = original_tokenizer.apply_chat_template(
        test_messages,
        tools=test_tools,
        tokenize=False,
        add_generation_prompt=True
    )
except Exception as e:
    print(f"Warning: Chat template test failed: {e}")

# Create a clean version of the merged model with correct chat template
merged_model_path = "./qwen3-function-calling-merged"
clean_model_path = "./qwen3-function-calling-merged-clean"

# Copy the merged model to clean directory
import shutil
if Path(clean_model_path).exists():
    shutil.rmtree(clean_model_path)
shutil.copytree(merged_model_path, clean_model_path)

# Update the tokenizer config with correct chat template
tokenizer_config_path = Path(clean_model_path) / "tokenizer_config.json"

with open(tokenizer_config_path, 'r') as f:
    config = json.load(f)

# Add the correct chat template
config['chat_template'] = correct_chat_template

# Save the updated config
with open(tokenizer_config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Clean model ready at: {clean_model_path}")

## GGUF Conversion for Edge Deployment

Convert the fine-tuned model to GGUF format for optimized edge deployment with llama.cpp.

#### Performance Optimizations:
- **Quantization**: Reduce model size by 75% with Q4_K_M quantization
- **Memory Mapping**: Instant model loading without RAM copying
- **SIMD Acceleration**: Optimized for modern CPU instruction sets

#### Deployment Advantages:
- **Single File**: Everything needed for inference in one file
- **Cross-Platform**: Works on x86, ARM, Qualcomm and embedded systems
- **No Dependencies**: Minimal runtime requirements

### Quantization Strategy

We use Q4_K_M quantization which provides:
- **75% Size Reduction**: From ~3.4GB to ~1.1GB
- **2-3x Speed Improvement**: Faster inference on edge hardware
- **<5% Accuracy Loss**: Minimal impact on function calling performance
- **Memory Efficiency**: Fits in 2GB RAM with room for context

In [ ]:
# GGUF conversion configuration
import subprocess
import shutil
from pathlib import Path

# Configuration - Use the CLEAN model with correct chat template
MODEL_NAME = "qwen3-function-calling"
QUANTIZATION = "Q4_K_M"  # Good balance of size and quality
merged_model_path = "./qwen3-function-calling-merged-clean"  # Use clean model!
output_name = f"{MODEL_NAME}-{QUANTIZATION}.gguf"
source_tokenizer_dir = "./qwen3-function-calling-final"

print(f"Converting {MODEL_NAME} to GGUF format with {QUANTIZATION} quantization")

### Execute GGUF Conversion

Run the conversion pipeline to create the quantized GGUF model for edge deployment.

In [ ]:
# Execute GGUF conversion and quantization
model_dir = Path(merged_model_path)
gguf_dir = model_dir.with_name(model_dir.name + "-gguf")
gguf_dir.mkdir(parents=True, exist_ok=True)
gguf_path = gguf_dir / output_name

# Check for required tokenizer files and copy if needed
required_files = ["tokenizer.json", "vocab.json", "tokenizer_config.json"]
missing_files = [f for f in required_files if not (model_dir / f).exists()]

if missing_files and Path(source_tokenizer_dir).exists():
    source_dir = Path(source_tokenizer_dir)
    for file in missing_files:
        source_file = source_dir / file
        if source_file.exists():
            shutil.copy2(source_file, model_dir / file)

# Verify conversion tools exist
convert_script = Path("./llama.cpp/convert_hf_to_gguf.py")
quantize_binary = Path("./llama.cpp/build/bin/llama-quantize")

if not convert_script.exists():
    print(f"Error: Conversion script not found at {convert_script}")
elif not quantize_binary.exists():
    print(f"Error: Quantization binary not found at {quantize_binary}")
else:
    # Step 1: Convert to FP16 GGUF
    temp_gguf = gguf_dir / "temp.gguf"
    
    try:
        subprocess.run([
            "python", str(convert_script),
            str(model_dir),
            "--outfile", str(temp_gguf),
            "--outtype", "f16"
        ], check=True, capture_output=True, text=True)
        
        # Step 2: Quantize to target format
        subprocess.run([
            str(quantize_binary),
            str(temp_gguf),
            str(gguf_path),
            QUANTIZATION
        ], check=True, capture_output=True, text=True)
        
        # Clean up temporary file
        if temp_gguf.exists():
            temp_gguf.unlink()
        
        # Verify output and report results
        if gguf_path.exists():
            file_size = gguf_path.stat().st_size / (1024**3)
            print(f"GGUF conversion completed: {gguf_path.name} ({file_size:.2f} GB)")
            
            # Store path for next steps
            GGUF_MODEL_PATH = str(gguf_path)
        else:
            print(f"Error: Output file not created at {gguf_path}")
            GGUF_MODEL_PATH = None
            
    except subprocess.CalledProcessError as e:
        print(f"Conversion failed: {e}")
        if e.stderr:
            print(f"Error details: {e.stderr}")
        # Clean up temp file if it exists
        if temp_gguf.exists():
            temp_gguf.unlink()
        GGUF_MODEL_PATH = None

## Model Test & Evaluation Harness

Evaluate and compare all three model formats to demonstrate the effectiveness of fine-tuning and quantization. Our evaluation compares three model variants across multiple dimensions:

#### Model Variants:
1. **Raw Base Model**: Original Qwen3-1.7B without any fine-tuning
2. **Fine-tuned Model**: LoRA-adapted model optimized for function calling
3. **GGUF Quantized Model**: Q4_K_M quantized version for edge deployment

#### Test Scenarios:
- **Climate Control**: Temperature, fan, and mode adjustments
- **Window Control**: Opening, closing, and positioning windows
- **Seat Control**: Position, heating, and massage adjustments
- **Lighting Control**: Headlights, ambient, and interior lighting
- **Drive Mode**: Sport, eco, and comfort mode switching

The evaluation will demonstrate that fine-tuning provides improvements in function calling capability, while quantization maintains most of this performance with significant efficiency gains.

In [ ]:
# Setup for model evaluation
import time
import json
import re
import requests
import subprocess
import psutil
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
print(torch.cuda.is_available())

# Test cases for evaluation
TEST_CASES = [
    {
        "prompt": "Set the temperature to 72 degrees",
        "expected_tool": "climate_control",  
        "expected_args": {"action": "set_temperature", "temperature": 72}
    },
    {
        "prompt": "Open the driver side window", 
        "expected_tool": "window_control", 
        "expected_args": {"action": "open", "target": "driver"}
    },
    {
        "prompt": "Switch to sport mode",
        "expected_tool": "drive_mode",  
        "expected_args": {"action": "set_mode", "mode": "sport"}
    },
    {
        "prompt": "Turn on the headlights",
        "expected_tool": "lighting_control", 
        "expected_args": {"action": "set_headlights", "headlight_mode": "on"}
    },
    {
        "prompt": "Move seat forward",
        "expected_tool": "seat_control",  
        "expected_args": {"action": "adjust_position", "seat": "driver", "position_type": "forward", "adjustment": 10}
    },
    {
        "prompt": "Close all windows",
        "expected_tool": "window_control",  
        "expected_args": {"action": "close", "target": "all"}
    },
    {
        "prompt": "Turn on seat heating",
        "expected_tool": "seat_control",
        "expected_args": {"action": "set_heating", "seat": "driver", "heating_level": 2}
    },
    {
        "prompt": "Set ambient lighting to blue",
        "expected_tool": "lighting_control", 
        "expected_args": {"action": "set_ambient", "ambient_color": "blue", "ambient_brightness": 60}
    }
]

print(f"Loaded {len(TEST_CASES)} test cases for evaluation")

## Evaluation Infrastructure Setup

Configure the evaluation framework with test cases, helper functions, and tool specifications. Our evaluation uses designed test cases that cover:

#### Comprehensive Coverage:
- **All Tool Types**: Each of the 5 vehicle control domains
- **Parameter Variety**: Different parameter combinations and values
- **Natural Language**: Realistic user requests with varied phrasing
- **Edge Cases**: Ambiguous requests and boundary conditions

In [ ]:
# Load tools dynamically from cockpit agents
import sys
import os

current_dir = os.getcwd() 
src_dir = os.path.dirname(current_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
    
from agents.cockpit import climate_control, window_control, seat_control, lighting_control, drive_mode

cockpit_tools = [climate_control.tool_spec, window_control.tool_spec, seat_control.tool_spec, lighting_control.tool_spec, drive_mode.tool_spec]

def convert_strands_tool_format(strands_tool):
    """Convert Strands tool format to OpenAI function format.
    
    This function converts the Strands tool_spec format to the OpenAI
    function calling format used by chat completions API.
    
    Args:
        strands_tool: Tool spec with 'name', 'description', and 'inputSchema' fields
        
    Returns:
        dict: OpenAI-compatible tool definition
    """
    return {
        "type": "function",
        "function": {
            "name": strands_tool["name"],
            "description": strands_tool["description"],
            "parameters": strands_tool["inputSchema"]["json"]
        }
    }

# Convert all cockpit tools to OpenAI format for API-based evaluations
# (Used by GGUF evaluation with llama.cpp server)
formatted_tools = [convert_strands_tool_format(tool) for tool in cockpit_tools]

# JSON payload template for function calling evaluation. This will be used with the Jinja template to generate prompts
PROMPT_JSON_TEMPLATE = {
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant with access to vehicle control tools."
        },
        {
            "role": "user",
            "content": "{user_message}"
        }
    ],
    "tools": formatted_tools, 
    "add_generation_prompt": True,
    "enable_thinking": False
}

In [ ]:
# Helper functions for evaluation
import re
import json
import time
import psutil
import jinja2
import copy
import sys
import os
from typing import Dict, List, Any
import inspect

# Load the Jinja template for chat formatting
def load_chat_template():
    templateLoader = jinja2.FileSystemLoader(searchpath="./utils")
    templateEnv = jinja2.Environment(loader=templateLoader)
    template = templateEnv.get_template("chat_template.jinja")
    return template

# Function to generate prompt using Jinja template
def generate_prompt_from_json(template, json_payload, user_message):
    # Create a copy of the template and substitute the user message
    payload = copy.deepcopy(json_payload)
    for message in payload['messages']:
        if message['role'] == 'user':
            message['content'] = message['content'].format(user_message=user_message)
    
    # Render the template
    return template.render(**payload)

# Extract tool call from model response
def extract_tool_call(response):
    try:
        match = re.search(r'<tool_call>(.*?)</tool_call>', response, re.DOTALL)
        if not match:
            return None, None
        
        tool_json_str = match.group(1).strip()
        
        tool_json = json.loads(tool_json_str)
        tool_name = tool_json.get('name')
        tool_args = tool_json.get('arguments')
        
        print(f"[extract_tool_call] Parsed - name: {tool_name}, args: {tool_args}")
        return tool_name, tool_args
    except Exception:
        return None, None

# Measure current memory usage
def measure_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

print("Evaluation setup complete")
print(f"Loaded {len(cockpit_tools)} tools and converted to OpenAI format")
print("Note: Raw/Fine-tuned models use Jinja templates, GGUF uses OpenAI format")

## Baseline Model Evaluation

Establish baseline performance by evaluating the raw Qwen3-1.7B model without any fine-tuning.

### Evaluation Process

The baseline evaluation:
1. **Loads Raw Model**: Original Qwen3-1.7B without modifications
2. **Tests Function Calling**: Attempts to generate tool calls for test cases
3. **Measures Performance**: Records accuracy, speed, and memory usage
4. **Establishes Baseline**: Creates reference point for improvement measurement

This baseline demonstrates the necessity and effectiveness of our fine-tuning approach.

In [ ]:
# Evaluate raw base model performance
start_mem = measure_memory_usage()

# Check CUDA availability and force GPU usage
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load raw base model with explicit device mapping
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B",
    torch_dtype=torch.float16,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

# If device_map="auto" didn't work, manually move to GPU
if torch.cuda.is_available() and base_model.device.type == "cpu":
    base_model = base_model.to(device)

base_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B", trust_remote_code=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

load_mem = measure_memory_usage()

# Initialize results tracking
raw_results = {
    "model_type": "Raw Base Model",
    "memory_usage_mb": load_mem - start_mem,
    "test_results": [],
    "total_time": 0,
    "accuracy": 0
}

base_model.eval()
correct_predictions = 0
total_time = 0

# Run evaluation on test cases
for i, test_case in enumerate(TEST_CASES):
    # Use PROMPT_JSON_TEMPLATE and update user message
    prompt_data = copy.deepcopy(PROMPT_JSON_TEMPLATE)
    # Update user message with test case prompt
    for message in prompt_data['messages']:
        if message['role'] == 'user':
            message['content'] = message['content'].format(user_message=test_case["prompt"])
    
    # Apply chat template with tools using OpenAI format
    prompt = base_tokenizer.apply_chat_template(
        prompt_data['messages'],
        tokenize=False,
        add_generation_prompt=True,
        tools=formatted_tools  # Use same OpenAI-formatted tools as GGUF
    )
    
    # Tokenize input
    inputs = base_tokenizer(prompt, return_tensors="pt", truncation=True)
    inputs = {k: v.to(base_model.device) for k, v in inputs.items()}
    
    # Time inference
    start_time = time.time()
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.1,
            do_sample=False,
            pad_token_id=base_tokenizer.pad_token_id
        )
    inference_time = time.time() - start_time
    total_time += inference_time
    
    # Decode response
    response = base_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    
    # Extract tool call
    tool_name, tool_args = extract_tool_call(response)
    
    
    # Check accuracy
    is_correct = tool_name == test_case["expected_tool"]
    if is_correct:
        correct_predictions += 1
    
    # Store results
    raw_results["test_results"].append({
        "test_case": test_case["prompt"],
        "expected_tool": test_case["expected_tool"],
        "predicted_tool": tool_name,
        "correct": is_correct,
        "inference_time_ms": inference_time * 1000,
        "response": response,
        "full_response": response
    })
    
    # Print test result
    status = "PASS" if is_correct else "FAIL"
    print(f"  Test {i+1}/8: {status} - {test_case['prompt'][:30]}... ({inference_time*1000:.0f}ms)")
    if not is_correct:
        print(f"    Expected: {test_case['expected_tool']}")
        print(f"    Got: {tool_name or 'None'}")

# Calculate final metrics
raw_results["accuracy"] = correct_predictions / len(TEST_CASES)
raw_results["total_time"] = total_time
raw_results["avg_time_ms"] = (total_time / len(TEST_CASES)) * 1000

print(f"\nRaw model evaluation complete:")
print(f"  Accuracy: {raw_results['accuracy']:.1%}")
print(f"  Average inference time: {raw_results['avg_time_ms']:.0f}ms")
print(f"  Memory usage: {raw_results['memory_usage_mb']:.0f}MB")

# Clean up
del base_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## GGUF Model Evaluation

Evaluate the performance of the quantized GGUF model using llama.cpp server.

**Updated Methodology**: This evaluation now uses the OpenAI-compatible chat completions API endpoint (`/v1/chat/completions`) with proper tool calling format, matching the production implementation in `evaluate_gguf_model.py`. This approach:

- Uses the same OpenAI format that Strands agents use successfully
- Leverages the embedded Jinja chat template for proper tool calling
- Converts Strands tool definitions to OpenAI function format
- Parses structured tool calls from the response

This ensures the evaluation accurately reflects real-world usage patterns.

In [ ]:
# GGUF Model Evaluation Setup
import subprocess
import requests
import signal
import os
import json
from pathlib import Path

# GGUF model path - updated to match current implementation
gguf_path = "./qwen3-function-calling-merged-clean-gguf/qwen3-function-calling-Q4_K_M.gguf"
server_port = 8081

# Check if GGUF file exists
if not Path(gguf_path).exists():
    print(f"GGUF file not found: {gguf_path}")
    print("Skipping GGUF evaluation")
    gguf_results = None
else:
    print(f"Starting GGUF server on port {server_port}...")
    
    # Start llama.cpp server with embedded chat template for proper tool calling
    server_cmd = [
        "./llama.cpp/build/bin/llama-server",
        "-m", gguf_path,
        "--port", str(server_port),
        "--host", "0.0.0.0",
        "--jinja"  # Use embedded Jinja template for tool calling
    ]
    print(f"Server command: {' '.join(server_cmd)}")
    
    try:
        # Start server process
        server_process = subprocess.Popen(
            server_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            preexec_fn=os.setsid
        )
        
        # Wait for server to be ready
        import time
        max_wait = 30
        wait_time = 0
        server_ready = False
        
        print("Waiting for server to start...")
        while wait_time < max_wait:
            try:
                response = requests.get(f"http://localhost:{server_port}/health", timeout=2)
                if response.status_code == 200:
                    server_ready = True
                    break
            except:
                pass
            time.sleep(1)
            wait_time += 1
            print(f"  Waiting... ({wait_time}/{max_wait}s)")
        
        if not server_ready:
            print("Server failed to start within timeout")
            gguf_results = None
        else:
            print("Server ready, starting evaluation...")
            
            # Initialize results tracking
            gguf_results = {
                "model_type": "GGUF Quantized Model",
                "memory_usage_mb": "N/A (External Server)",
                "test_results": [],
                "total_time": 0,
                "accuracy": 0
            }
            
            correct_predictions = 0
            total_time = 0
            
            print("\nRunning GGUF evaluation tests...")
            
            # Run evaluation on test cases
            for i, test_case in enumerate(TEST_CASES):
                print(f"\nProcessing test {i+1}/{len(TEST_CASES)}: {test_case['prompt']}")
                
                # Use PROMPT_JSON_TEMPLATE structure for consistency
                prompt_data = copy.deepcopy(PROMPT_JSON_TEMPLATE)
                # Update user message with test case prompt
                for message in prompt_data['messages']:
                    if message['role'] == 'user':
                        message['content'] = message['content'].format(user_message=test_case["prompt"])
                
                # Use OpenAI API format with same message structure
                payload = {
                    "model": "qwen3",
                    "messages": prompt_data['messages'],
                    "tools": formatted_tools,
                    "temperature": 0.1,
                    "max_tokens": 200,
                    "stream": False
                }
                
                # Time inference
                start_time = time.time()
                try:
                    response = requests.post(
                        f"http://localhost:{server_port}/v1/chat/completions",
                        json=payload,
                        timeout=60
                    )
                    response.raise_for_status()
                    result = response.json()
                    
                    # Parse OpenAI chat completions response format
                    choice = result.get("choices", [{}])[0]
                    message = choice.get("message", {})
                    tool_calls = message.get("tool_calls", [])
                    response_text = message.get("content", "")
                    
                    # Extract tool call from OpenAI format
                    if tool_calls:
                        tool_call = tool_calls[0]  # Take first tool call
                        function = tool_call.get("function", {})
                        tool_name = function.get("name", "")
                        try:
                            tool_args = json.loads(function.get("arguments", "{}"))
                        except:
                            tool_args = {}
                    else:
                        tool_name = ""
                        tool_args = {}
                        
                except Exception as e:
                    print(f"    Request failed: {e}")
                    response_text = ""
                    tool_name = ""
                    tool_args = {}
                    result = {}
                    tool_calls = []
                
                inference_time = time.time() - start_time
                total_time += inference_time
                
                
                # Check accuracy
                is_correct = tool_name == test_case["expected_tool"]
                if is_correct:
                    correct_predictions += 1
                
                # Store results
                gguf_results["test_results"].append({
                    "test_case": test_case["prompt"],
                    "expected_tool": test_case["expected_tool"],
                    "predicted_tool": tool_name,
                    "predicted_args": tool_args,
                    "correct": is_correct,
                    "inference_time_ms": inference_time * 1000,
                    "response_text": response_text,
                    "tool_calls": tool_calls if 'tool_calls' in locals() else []
                })
                
                # Print test result
                status = "PASS" if is_correct else "FAIL"
                print(f"  Test {i+1}/8: {status} - {test_case['prompt'][:30]}... ({inference_time*1000:.0f}ms)")
                if not is_correct:
                    print(f"    Expected: {test_case['expected_tool']}")
                    print(f"    Got: {tool_name or 'None'}")
            
            # Calculate final metrics
            gguf_results["accuracy"] = correct_predictions / len(TEST_CASES)
            gguf_results["total_time"] = total_time
            gguf_results["avg_time_ms"] = (total_time / len(TEST_CASES)) * 1000
            
            print(f"\nGGUF model evaluation complete:")
            print(f"  Accuracy: {gguf_results['accuracy']:.1%}")
            print(f"  Average inference time: {gguf_results['avg_time_ms']:.0f}ms")
        
    except Exception as e:
        print(f"GGUF evaluation failed: {e}")
        gguf_results = None
    
    finally:
        # Clean up server
        if 'server_process' in locals():
            print("Stopping GGUF server...")
            try:
                os.killpg(os.getpgid(server_process.pid), signal.SIGTERM)
                server_process.wait(timeout=5)
            except:
                pass

## Results Comparison

Compare the performance of all three model formats side by side.

In [ ]:
print("Model Evaluation Results Comparison")
print("=" * 50)

# Collect all results
all_results = []
if 'raw_results' in locals():
    all_results.append(raw_results)
if 'gguf_results' in locals() and gguf_results is not None:
    all_results.append(gguf_results)

if not all_results:
    print("No evaluation results available for comparison")
else:
    # Summary table
    print(f"{'Model Type':<25} {'Accuracy':<10} {'Avg Time (ms)':<15} {'Memory (MB)':<12}")
    print("-" * 70)
    
    for result in all_results:
        model_type = result['model_type']
        accuracy = f"{result['accuracy']:.1%}"
        avg_time = f"{result['avg_time_ms']:.0f}ms"
        memory = str(result['memory_usage_mb']) if isinstance(result['memory_usage_mb'], str) else f"{result['memory_usage_mb']:.0f}"
        
        print(f"{model_type:<25} {accuracy:<10} {avg_time:<15} {memory:<12}")
    
    # Performance insights
    print("\nKey Insights:")
    print("-" * 20)
    
    if len(all_results) >= 2:
        best_accuracy = max(r['accuracy'] for r in all_results)
        fastest_time = min(r['avg_time_ms'] for r in all_results)
        
        print(f"Best accuracy: {best_accuracy:.1%}")
        print(f"Fastest inference: {fastest_time:.0f}ms average")
        
        # Find best performing models
        best_accuracy_model = next(r for r in all_results if r['accuracy'] == best_accuracy)
        fastest_model = next(r for r in all_results if r['avg_time_ms'] == fastest_time)
        
        print(f"Most accurate model: {best_accuracy_model['model_type']}")
        print(f"Fastest model: {fastest_model['model_type']}")
    
    # Detailed test results
    print("\nDetailed Test Results:")
    print("-" * 30)
    
    for i, test_case in enumerate(TEST_CASES):
        print(f"\nTest {i+1}: {test_case['prompt']}")
        print(f"Expected: {test_case['expected_tool']}")
        
        for result in all_results:
            if i < len(result['test_results']):
                test_result = result['test_results'][i]
                status = "PASS" if test_result['correct'] else "FAIL"
                predicted = test_result['predicted_tool'] or "None"
                time_ms = test_result['inference_time_ms']
                
                print(f"  {result['model_type']:<25} {status:<4} {predicted:<15} ({time_ms:.0f}ms)")

print("\nEvaluation complete!")

## Upload GGUF Model to S3

Upload the production-ready GGUF model to S3 for fast edge deployment.
GGUF files are optimized for edge inference and much smaller than full model directories.


In [ ]:
# Upload GGUF model to S3 for edge deployment
import boto3
from pathlib import Path
from botocore.exceptions import ClientError, NoCredentialsError

boto3_session = boto3.session.Session()
aws_account_id = boto3.client("sts").get_caller_identity()["Account"]

# Configuration - Update these values for your deployment
S3_BUCKET_NAME = f"automotive-workshop-{aws_account_id}-{boto3_session.region_name}"  # Replace with your bucket name
S3_KEY = "gguf-models/qwen3-function-calling-Q4_K_M.gguf"  # S3 path for the model
GGUF_FILE_PATH = "./qwen3-function-calling-merged-clean-gguf/qwen3-function-calling-Q4_K_M.gguf"

print("Uploading GGUF Model to S3")
print("=" * 40)

try:
    # Initialize S3 client
    s3_client = boto3.client('s3')
    
    # Check if GGUF file exists
    gguf_path = Path(GGUF_FILE_PATH)
    if not gguf_path.exists():
        print(f"GGUF file not found: {GGUF_FILE_PATH}")
        print("Please ensure the GGUF conversion step completed successfully.")
    else:
        # Get file size for progress reporting
        file_size_mb = gguf_path.stat().st_size / 1024 / 1024
        print(f"File: {gguf_path.name}")
        print(f"Size: {file_size_mb:.1f} MB")
        print(f"Destination: s3://{S3_BUCKET_NAME}/{S3_KEY}")
        
        # Check if bucket is accessible
        try:
            s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
            print(f"S3 bucket '{S3_BUCKET_NAME}' is accessible")
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == '404':
                print(f"Bucket '{S3_BUCKET_NAME}' does not exist")
                print("Please create the bucket or update S3_BUCKET_NAME")
                raise
            else:
                print(f"Error accessing bucket: {e}")
                raise
        
        # Upload the file
        print("\nStarting upload...")
        s3_client.upload_file(
            str(gguf_path),
            S3_BUCKET_NAME,
            S3_KEY,
            ExtraArgs={
                'ContentType': 'application/octet-stream',
                'ServerSideEncryption': 'AES256'
            }
        )
        
        print(f"\nModel Details:")
        print(f"  S3 Location: s3://{S3_BUCKET_NAME}/{S3_KEY}")
        print(f"  File Size: {file_size_mb:.1f} MB")

except NoCredentialsError:
    print("AWS credentials not found.")
    
except Exception as e:
    print(f"Upload failed: {e}")